In [29]:
import logging
import requests
import json
import time
import pandas as pd
from habanero import Crossref
cr = Crossref()
from functools import partial
from tenacity import retry, stop_after_delay, wait_fixed, retry_if_exception_type

In [ ]:
retry_on_communication_error = partial(
    retry,
    stop=stop_after_delay(10),  # max. 10 seconds wait.
    wait=wait_fixed(0.4),  # wait 400ms 
    retry=retry_if_exception_type([([ConnectionError, TimeoutError])])
)()

logging.basicConfig(
    filename='DOI.log',
    filemode='w',
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logging.INFO
)

start_time = time.time()

In [ ]:
cr = Crossref(mailto="m.n.khanji@umcg.nl")


retry_on_communication_error = partial(
    retry,
    stop=stop_after_delay(10),  # max. 10 seconds wait.
    wait=wait_fixed(0.4),  # wait 400ms 
    retry=retry_if_exception_type([([ConnectionError, TimeoutError])])
)

logging.basicConfig(
    filename='DOI.log',
    filemode='w',
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logging.INFO
)

start_time = time.time()

csv_file = 'pmid_journal_doi_issn.csv'       #get csv file
df = pd.read_csv(csv_file)
dois = df.iloc[:, 2].tolist() # get dois from column (3rd in this case)

@retry_on_communication_error
def publisher_crossref_doi(dois):
    publishers = []
    for doi in dois:
        try:
            work = cr.works(ids=doi)
            publisher = work["message"].get("publisher")
            publishers.append(publisher)
        except Exception as e:
            logging.error(f"Error: API request failed for {doi}: {e}")
            publishers.append(None)
    return publishers

# Call function and assign result to a list
publisher_list = publisher_crossref_doi(dois)

# Add the publisher_list to the DataFrame as a new column
df['publisher'] = publisher_list

# Save the updated DataFrame to a new CSV file
df.to_csv('pmid_doi_journal_issn_publisher.csv', index=False)
logging.info("Done")

In [27]:
# Load the CSV file
nan = pd.read_csv('pmid_doi_journal_issn_publisher.csv')

# Identify the column with missing values
publisher = 'publisher'

# Find the missing values in the column
missing_values = nan[publisher].isnull()

# Create a new DataFrame with only the rows that have missing values
missing_df = nan[missing_values]
print(missing_df)

           pmid                        journal  \
48     15725418                      Brain Res   
61     18837167        Folia Microbiol (Praha)   
68     34353416             Biomed Environ Sci   
106    17487399                      Oncol Rep   
193    12894873  Arch Immunol Ther Exp (Warsz)   
...         ...                            ...   
84220  20569278                 J Cell Mol Med   
84283  34622637    Sheng Wu Gong Cheng Xue Bao   
84328  25941716                 Mikrobiologiia   
84332  20891039                 Klin Lab Diagn   
84407  17543180              Chin Med J (Engl)   

                                    doi       issn publisher  
48       10.1016/j.brainres.2004.11.056  0006-8993       NaN  
61                                  NaN  0015-5632       NaN  
68                  10.3967/bes2021.073  2214-0190       NaN  
106                                 NaN  1021-335X       NaN  
193                                 NaN  0004-069X       NaN  
...                  

In [ ]:
import requests
import json
import logging
import time
from tenacity import retry, stop_after_attempt, wait_exponential
import pandas as pd

logging.basicConfig(level=logging.INFO)

def get_publisher_id_from_issn(issn: str) -> str:
    """
    Query the CrossRef API for a single ISSN and return the publisher ID.
    """
    url = f"https://api.crossref.org/works?filter=issn:{issn}&select=publisher&mailto=m.n.khanji@umcg.nl"
    try:
        response = requests.get(url)
        response.raise_for_status()  # Raise an exception for bad status codes
        data = json.loads(response.text)
        if "message" in data and "items" in data["message"] and data["message"]["items"]:
            first_item = data["message"]["items"][0]
            if isinstance(first_item, dict):
                return list(first_item.values())[0]
    except requests.RequestException as e:
        logging.error(f"Error: API request failed for {issn}: {e}")
        return None

@retry(wait=wait_exponential(multiplier=1, min=4, max=10),
        stop=stop_after_attempt(3),
        reraise=True)
def get_publisher_ids_from_issn(missing_df: pd.DataFrame) -> list:
    """
    Read ISSNs from a the missing_df dataframe, query the CrossRef API, and return a list of publisher IDs.
    """
    issns = missing_df.iloc[:, 3].tolist()
    publisher_id_list = []
    for issn in issns:
        publisher_id = get_publisher_id_from_issn(issn)
        publisher_id_list.append(publisher_id)
        time.sleep(0.4)
    return publisher_id_list

# Create a new DataFrame with the original DataFrame adding publisher_ids list
new_df = pd.DataFrame(list(zip(missing_df['pmid'], missing_df['journal'], missing_df['doi'], missing_df['issn'], get_publisher_ids_from_issn(missing_df))), 
                    columns=['pmid', 'journal', 'doi', 'issn', 'publisher'])

# Export the new_df to a new CSV file called PMID_Publisher.csv
new_df.to_csv('publishers_FINAL.csv', index=False)

In [28]:
# Load the CSV file
fin = pd.read_csv('publishers_FINAL.csv')

# Identify the column with missing values
publisher_fin = 'publisher'

# Find the missing values in the column
final_missing_values = fin[publisher_fin].isnull()

# Create a new DataFrame with only the rows that have missing values
final_missing = fin[final_missing_values]
print(final_missing)

          pmid                               journal                  doi  \
2     34353416                    Biomed Environ Sci  10.3967/bes2021.073   
7     25593506                               Mol Vis                  NaN   
8     16448026  Proc IEEE Comput Syst Bioinform Conf                  NaN   
9     18357784                 Ukr Biokhim Zh (1999)                  NaN   
11    23705366                         Zhong Yao Cai                  NaN   
...        ...                                   ...                  ...   
2295  12974331             J Submicrosc Cytol Pathol                  NaN   
2297  17143789      Zhonghua Wei Chang Wai Ke Za Zhi                  NaN   
2298  15030085                     Afr J Med Med Sci                  NaN   
2299  30378342     Sichuan Da Xue Xue Bao Yi Xue Ban                  NaN   
2307  15969062           Sheng Wu Gong Cheng Xue Bao                  NaN   

           issn publisher  
2     2214-0190       NaN  
7     1090-0535    

In [32]:
df_combined = pd.concat([nan, fin], ignore_index=True)
print(len(df_combined), df_combined.head(5))

86728        pmid                   journal                               doi  \
0  28357617            Photosynth Res         10.1007/s11120-017-0371-1   
1  19024597                     APMIS  10.1111/j.1600-0463.2008.00999.x   
2  17543862                  Dev Cell      10.1016/j.devcel.2007.03.019   
3  35907859          Microb Cell Fact        10.1186/s12934-022-01877-3   
4  29087340  Proc Natl Acad Sci U S A           10.1073/pnas.1713574114   

        issn                                        publisher  
0  1573-5079          Springer Science and Business Media LLC  
1  0903-4641                                            Wiley  
2  1534-5807                                      Elsevier BV  
3  1475-2859          Springer Science and Business Media LLC  
4  1091-6490  Proceedings of the National Academy of Sciences  


In [34]:
final = df_combined.dropna(subset=['publisher'])
print(len(final))

83622


In [35]:
final.to_csv('FINAL_FINAL_85K.csv', index=False)